In [1]:
import numpy as np
import pandas as pd
import csv
import sys

#> --- API Imports ---
try:
    sys.path.insert(0, "..\\src\\k_prototype")
    from k_prototype import in_depth_encoder
except:
    sys.path.insert(0, "..\\src\\")
    from k_prototype import in_depth_encoder

In [12]:
#> --- Reading and Repersenting the Dataset --- <#
data_frame = pd.read_csv(r'..\data\raw\zomato_bangalore_restaurants.csv')
index_list=[i for i in range(1,len(data_frame)+1)]
data_frame.index=index_list
raw_data_amount=len(data_frame)

data_frame.iloc[:,] #> Showing some part of te dataframe

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
1,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
2,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
3,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
4,https://www.zomato.com/bangalore/addhuri-udupi...,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
5,https://www.zomato.com/bangalore/grand-village...,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51713,https://www.zomato.com/bangalore/best-brews-fo...,"Four Points by Sheraton Bengaluru, 43/3, White...",Best Brews - Four Points by Sheraton Bengaluru...,No,No,3.6 /5,27,080 40301477,Whitefield,Bar,NaN,Continental,"1,500","[('Rated 5.0', ""RATED\n Food and service are ...",[],Pubs and bars,Whitefield
51714,https://www.zomato.com/bangalore/vinod-bar-and...,"Number 10, Garudachar Palya, Mahadevapura, Whi...",Vinod Bar And Restaurant,No,No,NaN,0,+91 8197675843,Whitefield,Bar,NaN,Finger Food,600,[],[],Pubs and bars,Whitefield
51715,https://www.zomato.com/bangalore/plunge-sherat...,Sheraton Grand Bengaluru Whitefield Hotel & Co...,Plunge - Sheraton Grand Bengaluru Whitefield H...,No,No,NaN,0,NaN,Whitefield,Bar,NaN,Finger Food,"2,000",[],[],Pubs and bars,Whitefield
51716,https://www.zomato.com/bangalore/chime-sherato...,Sheraton Grand Bengaluru Whitefield Hotel & Co...,Chime - Sheraton Grand Bengaluru Whitefield Ho...,No,Yes,4.3 /5,236,080 49652769,"ITPL Main Road, Whitefield",Bar,"Cocktails, Pizza, Buttermilk",Finger Food,"2,500","[('Rated 4.0', 'RATED\n Nice and friendly pla...",[],Pubs and bars,Whitefield


In [13]:
del data_frame["url"], data_frame["phone"], data_frame["address"], data_frame["dish_liked"], data_frame["reviews_list"], data_frame["menu_item"]
# --- Cutting Irrelevant Parts Of Numbers --- #
data_frame["rate"] = data_frame["rate"].str.replace('/5', '')
data_frame["rate"] = data_frame["rate"].str.replace('. ', '')
data_frame["rate"] = data_frame["rate"].str.replace(' ', '')
# --- Replacing Every Untransformable Value to NaN --- #
data_frame["rate"] = pd.to_numeric(data_frame["rate"], errors='coerce')
# --- Replacing Undefiend Values with NaN --- #
data_frame[(data_frame["rate"] < 0) & (data_frame["rate"] > 5)] = np.nan
# --- Replacing The Column's Mean For NaN Values --- #
rate_array = data_frame["rate"].dropna().to_numpy()
data_frame["rate"] = data_frame["rate"].fillna(rate_array.mean())
# --- Clean-Up --- #
del rate_array

# --- Replacing Every Untransformable Value to NaN --- #
data_frame["votes"] = pd.to_numeric(data_frame["votes"], errors='coerce')
# --- Replacing Undefiend Values with NaN --- #
data_frame[(data_frame["votes"] < 0) & (data_frame["votes"] > 100000)] = np.nan
# --- Replacing The Column's Mean For NaN Values --- #
votes_array = data_frame["votes"].copy()
votes_array = votes_array.dropna()
data_frame["rate"].fillna(int(votes_array.mean()))
# --- Clean-Up --- #
del votes_array

# --- Cutting Irrelevant Parts Of Numbers --- #
data_frame["approx_cost(for two people)"] = data_frame["approx_cost(for two people)"].str.replace(' ', '')
data_frame["approx_cost(for two people)"] = data_frame["approx_cost(for two people)"].str.replace(',', '')
# --- Replacing Every Untransformable Value to NaN --- #
data_frame["approx_cost(for two people)"] = pd.to_numeric(data_frame["approx_cost(for two people)"], errors='coerce')
# --- Replacing Undefiend Values with NaN --- #
data_frame[(data_frame["approx_cost(for two people)"] < 0) & (data_frame["approx_cost(for two people)"] > 1000000)] = np.nan
# --- Replacing The Column's Mean For NaN Values --- #
approx_array = data_frame["approx_cost(for two people)"].copy()
approx_array = approx_array.dropna()
data_frame["approx_cost(for two people)"].fillna(int(approx_array.mean()))
# --- Clean-Up --- #
del approx_array

# --- Filling and Replacing 'nan' ---
data_frame["location"] = data_frame["location"].fillna("No Location")
data_frame["rest_type"] = data_frame["rest_type"].fillna("No Type")
data_frame["listed_in(city)"] = data_frame["listed_in(city)"].fillna("Not Listed")

data_frame[:10]

,name,online_order,book_table,rate,votes,location,rest_type,cuisines,approx_cost(for two people),listed_in(type),listed_in(city)
1,Jalsa,Yes,Yes,4.1,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800.0,Buffet,Banashankari
2,Spice Elephant,Yes,No,4.1,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800.0,Buffet,Banashankari
3,San Churro Cafe,Yes,No,3.8,918,Banashankari,"Cafe, Casual Dining","Cafe, Mexican, Italian",800.0,Buffet,Banashankari
4,Addhuri Udupi Bhojana,No,No,3.7,88,Banashankari,Quick Bites,"South Indian, North Indian",300.0,Buffet,Banashankari
5,Grand Village,No,No,3.8,166,Basavanagudi,Casual Dining,"North Indian, Rajasthani",600.0,Buffet,Banashankari
6,Timepass Dinner,Yes,No,3.8,286,Basavanagudi,Casual Dining,North Indian,600.0,Buffet,Banashankari
7,Rosewood International Hotel - Bar & Restaurant,No,No,3.6,8,Mysore Road,Casual Dining,"North Indian, South Indian, Andhra, Chinese",800.0,Buffet,Banashankari
8,Onesta,Yes,Yes,4.6,2556,Banashankari,"Casual Dining, Cafe","Pizza, Cafe, Italian",600.0,Cafes,Banashankari
9,Penthouse Cafe,Yes,No,4.0,324,Banashankari,Cafe,"Cafe, Italian, Continental",700.0,Cafes,Banashankari
10,Smacznego,Yes,No,4.2,504,Banashankari,Cafe,"Cafe, Mexican, Italian, Momos, Beverages",550.0,Cafes,Banashankari


In [14]:
# --- Replacing 0/1 Instead of Yes/No ---
data_frame["online_order"] = np.where(data_frame["online_order"].to_numpy()=='Yes', 1, 0)
data_frame["book_table"] = np.where(data_frame["book_table"].to_numpy()=='Yes', 1, 0)
# --- Normalizing Numerical Data ---
data_frame["rate"] = np.log1p(data_frame["rate"])
data_frame["votes"] = np.log1p(data_frame["votes"])
data_frame["approx_cost(for two people)"] = np.log1p(data_frame["approx_cost(for two people)"])
# --- Droping "nan" Values & Calculating Mean ---
normalized_rate_mean = data_frame["rate"].dropna().mean()
normalized_votes_mean = data_frame["votes"].dropna().mean()
normalized_approx_cost_mean = data_frame["approx_cost(for two people)"].dropna().mean()
# --- Replacing nan ---
data_frame["rate"] = data_frame["rate"].fillna(normalized_rate_mean)
data_frame["votes"] = data_frame["votes"].fillna(normalized_votes_mean)
data_frame["approx_cost(for two people)"] = data_frame["approx_cost(for two people)"].fillna(normalized_approx_cost_mean)

data_frame[:15]

,name,online_order,book_table,rate,votes,location,rest_type,cuisines,approx_cost(for two people),listed_in(type),listed_in(city)
1,Jalsa,1,1,1.629241,6.654153,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",6.685861,Buffet,Banashankari
2,Spice Elephant,1,0,1.629241,6.669498,Banashankari,Casual Dining,"Chinese, North Indian, Thai",6.685861,Buffet,Banashankari
3,San Churro Cafe,1,0,1.568616,6.823286,Banashankari,"Cafe, Casual Dining","Cafe, Mexican, Italian",6.685861,Buffet,Banashankari
4,Addhuri Udupi Bhojana,0,0,1.547563,4.488636,Banashankari,Quick Bites,"South Indian, North Indian",5.707110,Buffet,Banashankari
5,Grand Village,0,0,1.568616,5.117994,Basavanagudi,Casual Dining,"North Indian, Rajasthani",6.398595,Buffet,Banashankari
6,Timepass Dinner,1,0,1.568616,5.659482,Basavanagudi,Casual Dining,North Indian,6.398595,Buffet,Banashankari
7,Rosewood International Hotel - Bar & Restaurant,0,0,1.526056,2.197225,Mysore Road,Casual Dining,"North Indian, South Indian, Andhra, Chinese",6.685861,Buffet,Banashankari
8,Onesta,1,1,1.722767,7.846590,Banashankari,"Casual Dining, Cafe","Pizza, Cafe, Italian",6.398595,Cafes,Banashankari
9,Penthouse Cafe,1,0,1.609438,5.783825,Banashankari,Cafe,"Cafe, Italian, Continental",6.552508,Cafes,Banashankari
10,Smacznego,1,0,1.648659,6.224558,Banashankari,Cafe,"Cafe, Mexican, Italian, Momos, Beverages",6.311735,Cafes,Banashankari


In [15]:
# --- Encoding Categorical Values ---
location_encoded_values, location_decoder_map = in_depth_encoder(data_frame, "location")
rest_type_encoded_values, rest_type_decoder_map = in_depth_encoder(data_frame, "rest_type")
cuisines_encoded_values, cuisines_decoder_map = in_depth_encoder(data_frame, "cuisines")
listed_in_type_encoded_values, listed_in_type_decoder_map = in_depth_encoder(data_frame, "listed_in(type)")
listed_in_city_encoded_values, listed_in_city_decoder_map = in_depth_encoder(data_frame, "listed_in(city)")
# --- Replacing Values ---
data_frame["location"], data_frame["rest_type"], data_frame["cuisines"] = location_encoded_values, rest_type_encoded_values, cuisines_encoded_values
data_frame["listed_in(type)"], data_frame["listed_in(city)"] = listed_in_type_encoded_values, listed_in_city_encoded_values
# --- Saving Decoding Maps ---
np.save(r'..\\data\\decoder_maps\\location_decoder_map', location_decoder_map)
np.save('..\\data\\decoder_maps\\rest_type_decoder_map', rest_type_decoder_map)
np.save('..\\data\\decoder_maps\\cuisines_decoder_map', cuisines_decoder_map)
np.save('..\\data\\decoder_maps\\listed_in_type_decoder_map', listed_in_type_decoder_map)
np.save('..\\data\\decoder_maps\\listed_in_city_decoder_map', listed_in_city_decoder_map)

data_frame[:15]

,name,online_order,book_table,rate,votes,location,rest_type,cuisines,approx_cost(for two people),listed_in(type),listed_in(city)
1,Jalsa,1,1,1.629241,6.654153,1,29,1436,6.685861,0,1
2,Spice Elephant,1,0,1.629241,6.669498,1,29,1466,6.685861,0,1
3,San Churro Cafe,1,0,1.568616,6.823286,1,23,1191,6.685861,0,1
4,Addhuri Udupi Bhojana,0,0,1.547563,4.488636,1,66,1886,5.707110,0,1
5,Grand Village,0,0,1.568616,5.117994,4,29,1877,6.398595,0,1
6,Timepass Dinner,1,0,1.568616,5.659482,4,29,1870,6.398595,0,1
7,Rosewood International Hotel - Bar & Restaurant,0,0,1.526056,2.197225,57,29,203,6.685861,0,1
8,Onesta,1,1,1.722767,7.846590,1,23,1194,6.398595,1,1
9,Penthouse Cafe,1,0,1.609438,5.783825,1,22,1146,6.552508,1,1
10,Smacznego,1,0,1.648659,6.224558,1,22,724,6.311735,1,1


In [16]:
# --- Seperating Categorical & Numerical Values ---
categorical_data_frame = data_frame[["online_order", "book_table", "location", "rest_type", "cuisines", "listed_in(type)", "listed_in(city)"]]
numerical_data_frame = data_frame[["rate", "votes", "approx_cost(for two people)"]]
# --- Saving Decoding Maps ---
data_frame.to_csv(r'..\\data\\processed\\processed_and_encoded_zomato_bangalore_restaurants.csv', index=False)
categorical_data_frame.to_csv(r'..\\data\\processed\\seperated_and_encoded_categorical_features.csv', index=False)
numerical_data_frame.to_csv(r'..\\data\\processed\\seperated_and_encoded_numerical_features.csv', index=False)
